## Google Colab Setup

This notebook can run either locally from the repo or on Google Colab.

Colab workflow:
- Mount Google Drive and save all trained-model outputs there immediately.
- Download only the exact code, data, and pre-computed epitope prediction files needed
  into `/content/XAllergen2.0`.
- Download a fresh ESM-2 snapshot from Hugging Face.

Required GitHub-hosted artifacts:
- `models/baseline_frozen_esm2.pt`
- `data/epitopepredict_tcell/deepalgpro_train_tepitope_raw_predictions.csv.gz`
- `data/epitopepredict_tcell/deepalgpro_test_tepitope_raw_predictions.csv.gz`

Section C reads `results/rsa_regularization/sweep_summary.csv` (notebook 12),
`results/ss3_regularization/sweep_summary.csv` (notebook 13), and optionally
`results/ss3_high_lambda/sweep_summary.csv` (notebook 15).
Run those notebooks first and keep Drive mounted.

**Regularization formula**

$$\mathcal{L} = \lambda_\text{cls}\mathcal{L}_\text{cls}
  + \lambda_\text{reg}\,\mathbf{1}[y=1]\,\frac{1}{L}\sum_{i=1}^{L}\alpha_i(1-e_i)$$

where $e_i \in [0,1]$ is the per-residue T-cell epitope score from TEpitope
(max-pooled across all 7 HLA-DR alleles and all overlapping 15-mer windows,
then per-protein min-max normalized). The $\mathbf{1}[y=1]$ masking is enforced
implicitly: non-allergen proteins receive no epitope feature vector and are
excluded from the regularization loss.

In [1]:
import os
import sys
import urllib.request
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
os.environ['XALLERGEN_RUN_TARGET'] = 'colab' if IS_COLAB else 'local'

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/XAllergen2.0')
else:
    DRIVE_ROOT = None

if IS_COLAB:
    RUNTIME_ROOT = Path('/content/XAllergen2.0')
else:
    for _candidate in [Path.cwd(), *Path.cwd().parents]:
        if (_candidate / 'src' / 'xallergen').exists():
            RUNTIME_ROOT = _candidate
            break
    else:
        raise FileNotFoundError('Could not locate repo root with src/xallergen')

SRC_DIR             = RUNTIME_ROOT / 'src'
PACKAGE_DIR         = SRC_DIR / 'xallergen'
DATA_DIR            = RUNTIME_ROOT / 'data'
EPITOPE_DIR         = DATA_DIR / 'epitopepredict_tcell'
RUNTIME_MODELS_DIR  = RUNTIME_ROOT / 'models'
RUNTIME_RESULTS_DIR = RUNTIME_ROOT / 'results'
HF_DIR              = RUNTIME_ROOT / 'hf_models' / 'facebook_esm2_t6_8M_UR50D'

if IS_COLAB:
    MODELS_DIR  = DRIVE_ROOT / 'models'
    RESULTS_DIR = DRIVE_ROOT / 'results'
else:
    MODELS_DIR  = RUNTIME_MODELS_DIR
    RESULTS_DIR = RUNTIME_RESULTS_DIR

for path in [
    SRC_DIR, PACKAGE_DIR, DATA_DIR, EPITOPE_DIR,
    RUNTIME_MODELS_DIR, RUNTIME_RESULTS_DIR, MODELS_DIR, RESULTS_DIR, HF_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

if IS_COLAB:
    from huggingface_hub import snapshot_download

    RAW = 'https://raw.githubusercontent.com/Jeffateth/XAllergen2.0/main'

    package_files = [
        '__init__.py',
        'baseline_notebook_utils.py',
        'baseline_sasa_experiment.py',
        'epitope_preprocessing.py',
        'mtl_epitope_notebook_utils.py',
        'rsa_preprocessing.py',
    ]
    data_files = [
        'deepalgpro_train_cleaned.csv',
        'deepalgpro_test_cleaned.csv',
        'positives_splitB.csv',
    ]
    epitope_files = [
        'deepalgpro_train_tepitope_raw_predictions.csv.gz',
        'deepalgpro_test_tepitope_raw_predictions.csv.gz',
    ]

    download_jobs = []
    download_jobs.extend((f'{RAW}/src/xallergen/{name}', PACKAGE_DIR / name) for name in package_files)
    download_jobs.extend((f'{RAW}/data/{name}', DATA_DIR / name) for name in data_files)
    download_jobs.extend((f'{RAW}/data/epitopepredict_tcell/{name}', EPITOPE_DIR / name) for name in epitope_files)
    download_jobs.append((f'{RAW}/models/baseline_frozen_esm2.pt', RUNTIME_MODELS_DIR / 'baseline_frozen_esm2.pt'))

    failed_urls = []
    for url, dst in download_jobs:
        try:
            urllib.request.urlretrieve(url, dst)
        except Exception as exc:
            failed_urls.append((url, str(exc)))

    if failed_urls:
        details = '\n'.join(f'  - {url} -> {err}' for url, err in failed_urls)
        raise RuntimeError('Failed to download required GitHub artifacts:\n' + details)

    fresh_model_path = snapshot_download(
        repo_id='facebook/esm2_t6_8M_UR50D',
        local_dir=HF_DIR,
        local_dir_use_symlinks=False,
        force_download=True,
        resume_download=False,
    )
    os.environ['XALLERGEN_HF_MODEL_DIR'] = str(fresh_model_path)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'RUN_TARGET: {os.environ["XALLERGEN_RUN_TARGET"]}')
print(f'Runtime root: {RUNTIME_ROOT}')
print(f'Epitope dir:  {EPITOPE_DIR}')
print(f'Output results dir: {RESULTS_DIR}')

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

RUN_TARGET: colab
Runtime root: /content/XAllergen2.0
Epitope dir:  /content/XAllergen2.0/data/epitopepredict_tcell
Output results dir: /content/drive/MyDrive/XAllergen2.0/results


# Epitope Attention Regularization Sweep

Tests whether penalising pooling-attention on non-epitope residues improves allergenicity
classification and residue-level epitope localisation.

**Epitope features (`e_i`)** are derived from TEpitope predictions over 7 HLA-DR alleles
(peptide length 15, step 1). Per residue:
1. Collect raw log-odds scores from all overlapping 15-mers (all alleles).
2. Max-pool across alleles and windows; clip negatives to 0.
3. Normalize per-protein to [0, 1].

The regularization loss applies **only to allergen proteins** (`y = 1`). Non-allergens
receive no epitope feature vector and are masked from the loss automatically.

Results are saved to `results/epitope_regularization/` and compared against the
baseline and SS3-structured sweep in Section C.

In [2]:
import importlib
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'pyproject.toml').exists() or (candidate / 'src' / 'xallergen').exists():
            return candidate
    raise FileNotFoundError('Could not find repo root')


if 'DATA_DIR' not in globals():
    RUNTIME_ROOT   = _find_repo_root(Path.cwd())
    SRC_DIR        = RUNTIME_ROOT / 'src'
    DATA_DIR       = RUNTIME_ROOT / 'data'
    EPITOPE_DIR    = DATA_DIR / 'epitopepredict_tcell'
    RUNTIME_MODELS_DIR = RUNTIME_ROOT / 'models'
    RESULTS_DIR    = RUNTIME_ROOT / 'results'
    if str(SRC_DIR) not in sys.path:
        sys.path.insert(0, str(SRC_DIR))

missing_modules = []
for module_name in ('numpy', 'pandas', 'torch', 'IPython', 'sklearn'):
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_modules.append(module_name)

if missing_modules:
    raise ModuleNotFoundError(
        'Missing kernel dependencies: ' + ', '.join(missing_modules) +
        '. Run `make setup` and select the `Python (xallergen2)` kernel.'
    )

import pandas as pd
import torch
from IPython.display import display
from sklearn.model_selection import train_test_split

from xallergen.baseline_notebook_utils import RANDOM_STATE, seed_everything
from xallergen.baseline_sasa_experiment import (
    RSARegularizationConfig,
    run_lambda_rsa_sweep,
)
from xallergen.epitope_preprocessing import (
    inspect_epitope_inputs,
    load_epitope_lookup_dicts,
)

TRAIN_CSV                = DATA_DIR / 'deepalgpro_train_cleaned.csv'
TEST_CSV                 = DATA_DIR / 'deepalgpro_test_cleaned.csv'
POSITIVES_SPLITB_CSV     = DATA_DIR / 'positives_splitB.csv'
BASELINE_CHECKPOINT_PATH = RUNTIME_MODELS_DIR / 'baseline_frozen_esm2.pt'

TRAIN_EPITOPE_PATH = EPITOPE_DIR / 'deepalgpro_train_tepitope_raw_predictions.csv.gz'
TEST_EPITOPE_PATH  = EPITOPE_DIR / 'deepalgpro_test_tepitope_raw_predictions.csv.gz'

EPITOPE_RESULTS_DIR = RESULTS_DIR / 'epitope_regularization'
EPITOPE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
EPITOPE_SWEEP_CSV = EPITOPE_RESULTS_DIR / 'sweep_summary.csv'

for required_path in [
    TRAIN_CSV, TEST_CSV, TRAIN_EPITOPE_PATH, TEST_EPITOPE_PATH,
    POSITIVES_SPLITB_CSV, BASELINE_CHECKPOINT_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(f'Missing required file: {required_path}')

# λ=0 (unregularized baseline) is skipped — already present in the RSA sweep.
EPITOPE_CONFIG = RSARegularizationConfig(
    lambda_cls=1.0,
    lambda_rsa_values=(0.1, 0.5, 1.0, 5.0),
    add_special_tokens=False,
    feature_key='epitope',
)

seed_everything(RANDOM_STATE)
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Feature: epitope (TEpitope score, max-pooled + per-protein min-max norm)')
print(f'λ values: {EPITOPE_CONFIG.lambda_rsa_values}')
print(f'Output dir: {EPITOPE_RESULTS_DIR}')

Device: cuda
Feature: epitope (TEpitope score, max-pooled + per-protein min-max norm)
λ values: (0.1, 0.5, 1.0, 5.0)
Output dir: /content/drive/MyDrive/XAllergen2.0/results/epitope_regularization


---
## Section A — Data Loading and Epitope Feature Inspection

In [3]:
train_df = pd.read_csv(TRAIN_CSV).copy()
test_df  = pd.read_csv(TEST_CSV).copy()
for df in [train_df, test_df]:
    df['sequence_id'] = df['sequence_id'].astype(str).str.strip()
    df['sequence']    = df['sequence'].astype(str).str.strip().str.upper()
    df['label']       = df['label'].astype(int)

print(f'Train: {len(train_df):,} proteins  ({(train_df["label"]==1).sum():,} allergens)')
print(f'Test:  {len(test_df):,}  proteins  ({(test_df["label"]==1).sum():,} allergens)')

Train: 5,551 proteins  (2,723 allergens)
Test:  1,377  proteins  (672 allergens)


In [4]:
# Summary: prediction row counts, score/rank ranges, allergen coverage
display(inspect_epitope_inputs(
    train_predictions_path=TRAIN_EPITOPE_PATH,
    test_predictions_path=TEST_EPITOPE_PATH,
    train_frame=train_df,
    test_frame=test_df,
    mode='score',
))

,path,split,n_prediction_rows,n_unique_proteins,n_allergens_frame,coverage,score_min,score_max,rank_min,rank_max,mode,rank_threshold
0,/content/XAllergen2.0/data/epitopepredict_tcel...,train,4087832,2677,2723,0.9831,-10.0,10.00,1.0,919.0,score,None
1,/content/XAllergen2.0/data/epitopepredict_tcel...,test,1071546,661,672,0.9836,-10.0,9.82,1.0,832.0,score,None


In [5]:
seed_everything(RANDOM_STATE)
train_split_df, val_split_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=RANDOM_STATE,
    stratify=train_df['label'],
)
train_split_df = train_split_df.reset_index(drop=True).copy()
val_split_df   = val_split_df.reset_index(drop=True).copy()

print(f'Train: {len(train_split_df):,} | Val: {len(val_split_df):,} | Test: {len(test_df):,}')

Train: 4,995 | Val: 556 | Test: 1,377


In [6]:
# Build per-residue epitope feature vectors.
# Non-allergen proteins receive None and are automatically excluded from the loss.
print('Building epitope feature lookup dicts (may take ~30 s for 4 M rows)...')
train_epitope_lookup, test_epitope_lookup, lookup_summary = load_epitope_lookup_dicts(
    train_predictions_path=TRAIN_EPITOPE_PATH,
    test_predictions_path=TEST_EPITOPE_PATH,
    train_frame=train_df,
    test_frame=test_df,
    add_special_tokens=EPITOPE_CONFIG.add_special_tokens,
    mode='score',
)
display(pd.DataFrame([lookup_summary['train'], lookup_summary['test']]))

# Quick sanity check: verify a known allergen has a non-None vector
sample_allergen = train_df.loc[train_df['label'] == 1, 'sequence_id'].iloc[0]
vec = train_epitope_lookup[sample_allergen]
print(f'\nSample allergen: {sample_allergen}')
print(f'  Vector shape:   {vec.shape}')
print(f'  Score range:    [{float(vec.min()):.4f}, {float(vec.max()):.4f}]')
print(f'  Fraction > 0:   {float((vec > 0).float().mean()):.3f}')

# Verify a non-allergen gets None
sample_non_allergen = train_df.loc[train_df['label'] == 0, 'sequence_id'].iloc[0]
assert train_epitope_lookup[sample_non_allergen] is None, 'Non-allergen should have None'
print(f'\nNon-allergen {sample_non_allergen}: None ✓ (will be excluded from loss)')

Building epitope feature lookup dicts (may take ~30 s for 4 M rows)...


,split,n_total_proteins,n_allergens,n_with_epitope_vec,coverage_allergens,mode,predictions_path
0,train,5551,2723,2677,0.9831,score,/content/XAllergen2.0/data/epitopepredict_tcel...
1,test,1377,672,661,0.9836,score,/content/XAllergen2.0/data/epitopepredict_tcel...



Sample allergen: allergen_2182
  Vector shape:   torch.Size([133])
  Score range:    [0.1937, 1.0000]
  Fraction > 0:   1.000

Non-allergen non-allergen_3154: None ✓ (will be excluded from loss)


---
## Section B — Epitope Regularization Sweep

λ values: **0.1, 0.5, 1.0, 5.0**.  
Each run starts from the frozen ESM-2 baseline checkpoint.
Non-allergens are excluded from the regularization term automatically.

In [ ]:
sweep_df = run_lambda_rsa_sweep(
    config=EPITOPE_CONFIG,
    train_split_df=train_split_df,
    val_split_df=val_split_df,
    test_df=test_df,
    train_rsa_lookup=train_epitope_lookup,
    test_rsa_lookup=test_epitope_lookup,
    positives_splitb_csv=POSITIVES_SPLITB_CSV,
    output_dir=EPITOPE_RESULTS_DIR,
    device=DEVICE,
).loc[:, [
    'lambda_rsa', 'epoch',
    'val_auroc', 'val_precision', 'val_recall', 'val_f1', 'val_mcc', 'val_accuracy',
    'test_auroc', 'test_precision', 'test_recall', 'test_f1', 'test_mcc', 'test_accuracy',
    'residue_auroc', 'residue_auprc', 'residue_precision_at_k',
]].copy()
sweep_df.to_csv(EPITOPE_SWEEP_CSV, index=False)
display(sweep_df)
print(f'Saved to: {EPITOPE_SWEEP_CSV}')
torch.cuda.empty_cache()

In [8]:
best = sweep_df.loc[sweep_df['val_mcc'].idxmax()]
print('=== Epitope regularization: best λ by val MCC ===')
print(f'  Best λ       : {best["lambda_rsa"]}')
print(f'  Val MCC      : {best["val_mcc"]:.4f}')
print(f'  Test MCC     : {best["test_mcc"]:.4f}')
print(f'  Test AUROC   : {best["test_auroc"]:.4f}')
print(f'  Residue AUROC: {best["residue_auroc"]:.4f}')

=== Epitope regularization: best λ by val MCC ===
  Best λ       : 1.0
  Val MCC      : 0.8562
  Test MCC     : 0.8306
  Test AUROC   : 0.9697
  Residue AUROC: 0.4705


---
## Section C — Comparison Table

- **Baseline (λ=0)** — unregularized model from the RSA sweep (notebook 12)
- **RSA** — best λ > 0 by val MCC (notebook 12)
- **SS3-structured** — best λ by val MCC across notebooks 13 + 15
- **Epitope** — best λ by val MCC from this notebook

In [ ]:
RSA_SWEEP_CSV = RESULTS_DIR / 'rsa_regularization' / 'sweep_summary.csv'
SS3_SWEEP_CSV = RESULTS_DIR / 'ss3_regularization' / 'sweep_summary.csv'
SS3_HI_SWEEP_CSV = RESULTS_DIR / 'ss3_high_lambda' / 'sweep_summary.csv'

for p in [RSA_SWEEP_CSV, SS3_SWEEP_CSV]:
    if not p.exists():
        raise FileNotFoundError(f'{p} not found — run notebooks 12 and 13 first.')


def _row(csv_path, label, select_best=True):
    df  = pd.read_csv(csv_path)
    row = df.loc[df['val_mcc'].idxmax()] if select_best else df.loc[df['lambda_rsa'].eq(0.0)].iloc[0]
    return {
        'Constraint':        label,
        'Best λ':            row['lambda_rsa'],
        'Val MCC':           round(float(row['val_mcc']),              4),
        'Test MCC':          round(float(row['test_mcc']),             4),
        'Test AUROC':        round(float(row['test_auroc']),           4),
        'Residue AUROC':     round(float(row['residue_auroc']),        4),
        'Residue AUPRC':     round(float(row['residue_auprc']),        4),
        'Residue Prec@k':    round(float(row['residue_precision_at_k']), 4),
    }


def _row_from_df(df, label):
    row = df.loc[df['val_mcc'].idxmax()]
    return {
        'Constraint':        label,
        'Best λ':            row['lambda_rsa'],
        'Val MCC':           round(float(row['val_mcc']),              4),
        'Test MCC':          round(float(row['test_mcc']),             4),
        'Test AUROC':        round(float(row['test_auroc']),           4),
        'Residue AUROC':     round(float(row['residue_auroc']),        4),
        'Residue AUPRC':     round(float(row['residue_auprc']),        4),
        'Residue Prec@k':    round(float(row['residue_precision_at_k']), 4),
    }


# Merge SS3 sweeps if the high-lambda results exist
ss3_df = pd.read_csv(SS3_SWEEP_CSV)
if SS3_HI_SWEEP_CSV.exists():
    ss3_df = pd.concat([ss3_df, pd.read_csv(SS3_HI_SWEEP_CSV)], ignore_index=True)

comparison_df = pd.DataFrame([
    _row(RSA_SWEEP_CSV,    'Baseline (λ=0)',    select_best=False),
    _row(RSA_SWEEP_CSV,    'RSA',               select_best=True),
    _row_from_df(ss3_df,   'SS3-structured'),
    _row_from_df(sweep_df, 'Epitope'),
]).set_index('Constraint')

print('=== Cross-sweep comparison (best λ by val MCC) ===')
display(comparison_df)

=== Cross-sweep comparison (best λ by val MCC) ===


,Best λ,Val MCC,Test MCC,Test AUROC,Residue AUROC,Residue AUPRC,Residue Prec@k
Constraint,,,,,,,
Baseline (λ=0),0.0,0.8454,0.8336,0.9699,0.4677,0.2529,0.2163
RSA,0.5,0.8597,0.8474,0.9697,0.4660,0.2519,0.2148
SS3-structured,5.0,0.8777,0.8446,0.9684,0.5300,0.2863,0.2649
Epitope,1.0,0.8562,0.8306,0.9697,0.4705,0.2539,0.2167


: 

In [ ]:
# Shut down the Colab runtime to free GPU after training completes.
import os
os.kill(os.getpid(), 9)

: 

: 